# Function 3: Black-box optimisation

This notebook documents 13 rounds of model fitting, candidate selection and evaluated outputs for Function 3. Alternative approaches are retained where they informed the weekly decisions.

**Main result:** Local refinement improved the best output to −0.000123351 in Week 13.

**Usage:** Run the setup and then the weekly sections in order. The initial datasets are in this folder; the full query history is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv). Some early searches generate large candidate grids and may require substantial memory.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.optimize import differential_evolution
from scipy.spatial import cKDTree
from scipy.stats import norm, truncnorm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF


def resolve_initial_path(filename: str) -> Path:
    """Find the initial dataset for Function 3."""
    for folder in (Path.cwd(), Path.cwd() / "function_3"):
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"{filename} not found. Place it in the function_3/ folder."
    )

## Reusable routines

The helper functions below handle standard GP fitting, UCB scoring and recording evaluated queries. The model settings and experiments remain visible within each week. Global/local candidate generation is shared across the early rounds.

In [ ]:
def fit_gp_model(X, y, **settings):
    """Fit a GP using the explicit kernel and settings chosen for a given week."""
    gp = GaussianProcessRegressor(**settings)
    gp.fit(X, y)
    return gp


def ucb_score(mean, std, beta):
    """Upper Confidence Bound: predictive mean plus weighted uncertainty."""
    return mean + beta * std


def add_evaluation(X, y, x_observed, y_observed):
    """Add one completed black-box evaluation to the training history."""
    x_row = np.asarray(x_observed, dtype=float).reshape(1, -1)
    y_value = np.asarray(y_observed, dtype=float).reshape(1)
    if x_row.shape[1] != X.shape[1]:
        raise ValueError("The evaluated input has the wrong number of dimensions.")
    return np.vstack([X, x_row]), np.concatenate([y, y_value])

def sample_hybrid_candidates(best_x, n_global, n_local, seed, *,
                             local_scale=0.08, local_method="truncated"):
    """Combine uniform global samples with local samples near the current best.

    The local method is either "truncated" (truncated normal, inside [0, 1])
    or "clipped" (normal samples clipped to [0, 1]). Random number generation
    follows the same sequence for a given seed as the weekly experiments.
    """
    best_x = np.asarray(best_x, dtype=float).reshape(-1)
    n_features = best_x.size
    rng = np.random.default_rng(seed)

    X_global = rng.uniform(0, 1, size=(n_global, n_features))
    if local_method == "clipped":
        X_local = rng.normal(
            loc=best_x, scale=local_scale, size=(n_local, n_features)
        )
        X_local = np.clip(X_local, 0, 1)
    elif local_method == "truncated":
        X_local = np.empty((n_local, n_features))
        for j in range(n_features):
            a = (0 - best_x[j]) / local_scale
            b = (1 - best_x[j]) / local_scale
            X_local[:, j] = truncnorm.rvs(
                a, b, loc=best_x[j], scale=local_scale,
                size=n_local, random_state=rng
            )
    else:
        raise ValueError("local_method must be 'clipped' or 'truncated'.")

    return np.round(np.vstack([X_global, X_local]), 6)


## Starting data

In [2]:
X = np.load(resolve_initial_path("initial_inputs.npy"))
y = np.load(resolve_initial_path("initial_outputs.npy"))

## Week 1

In [6]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

# UCB exploration parameter
beta = 1.96
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds="fixed"
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True
)
# Generate candidate inputs
# Define the search range for each input variable

x1_values = np.linspace(0, 1, 100)
x2_values = np.linspace(0, 1, 100)
x3_values = np.linspace(0, 1, 100)

# Generate every combination of x1 and x2
X1_grid, X2_grid, X3_grid = np.meshgrid(
    x1_values,
    x2_values,
    x3_values
)

X_grid = np.column_stack([
    X1_grid.ravel(),
    X2_grid.ravel(),
    X3_grid.ravel()
])
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_grid,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)
# Select the next query
best_index = np.argmax(UCB)

next_X = X_grid[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.48484848 0.66666667 0.18181818]

Predicted mean:
-0.07538861801415585

Predicted standard deviation:
0.07903113470635567

UCB:
0.07951240601030125


### Evaluated query

**Input:** `(0.484848, 0.666667, 0.181818)`  
**Output:** `-0.12085975734719312`

In [ ]:
X_week1 = np.array([0.484848, 0.666667, 0.181818])
y_week1 = np.array([-0.12085975734719312])
X_new, y_new = add_evaluation(X, y, X_week1, y_week1)

## Week 2

In [10]:
X = X_new
y = y_new

### Increase β to favour exploration

In [11]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 2.5

### Allow the kernel length scale to vary

In [12]:
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.138)


### Use random candidates instead of a grid

In [13]:
# Generate candidate inputs
n_candidates = 1000000
n_features = 3

np.random.seed(42)

X_candidates = np.random.uniform(
    low=0,
    high=1,
    size=(n_candidates, n_features)
)

X_candidates = np.round(X_candidates, 6)

In [14]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [15]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.399048 0.467509 0.447396]

Predicted mean:
-0.05605545767708988

Predicted standard deviation:
0.07409651467108519

UCB:
0.1291858290006231


### Evaluated query

**Input:** `(0.393411, 0.462472, 0.461293)`  
**Output:** `-0.012120662033579272`

In [ ]:
X_week2 = np.array([0.393411, 0.462472, 0.461293])
y_week2 = np.array([-0.012120662033579272])
X_new, y_new = add_evaluation(X, y, X_week2, y_week2)

## Week 3

In [19]:
X = X_new
y = y_new

### Option 1: Isotropic GP with global/local sampling · selected

In [20]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 2.5

In [21]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.163)


In [22]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 3
n_global = 800000
n_local = 200000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="clipped",
)

In [23]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [24]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.478377 0.674274 0.551601]

Predicted mean:
-0.04534385041288426

Predicted standard deviation:
0.0721453604573703

UCB:
0.1350195507305415


### Option 2: Anisotropic GP with global/local sampling

In [25]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1]

beta = 2.5

In [26]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.01, 2.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[2, 0.792, 0.0574])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [27]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 3
n_global = 800000
n_local = 200000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="clipped",
)

In [28]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [29]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.95787  0.995954 0.418157]

Predicted mean:
-0.0013198618542200197

Predicted standard deviation:
0.0538364069667978

UCB:
0.13327115556277447


### Evaluated query

**Input:** `(0.478377, 0.674274, 0.551601)`  
**Output:** `-0.031644553454216365`

In [ ]:
X_week3 = np.array([0.478377, 0.674274, 0.551601])
y_week3 = np.array([-0.031644553454216365])
X_new, y_new = add_evaluation(X, y, X_week3, y_week3)

## Week 4

In [33]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 0.5 · selected

In [34]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 0.5

In [35]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.168)


In [36]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 3
n_global = 800000
n_local = 200000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [37]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [38]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.482994 0.557918 0.450225]

Predicted mean:
0.0019457280675653499

Predicted standard deviation:
0.025023547751809292

UCB:
0.014457501943469996


### Option 2: Anisotropic GP with β = 0.5 · alternative

In [39]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1]

beta = 0.5

In [40]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.01, 2.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[2, 0.79, 0.0589])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [41]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 3
n_global = 800000
n_local = 200000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [42]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [43]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.662814 0.988835 0.480364]

Predicted mean:
0.012921957923163807

Predicted standard deviation:
0.03563737138776781

UCB:
0.030740643617047713


### Evaluated query

**Input:** `(0.482994, 0.557918, 0.450225)`  
**Output:** `-0.0001325889446007488`

In [ ]:
X_week4 = np.array([0.482994, 0.557918, 0.450225])
y_week4 = np.array([-0.0001325889446007488])
X_new, y_new = add_evaluation(X, y, X_week4, y_week4)

## Week 5

In [47]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 0.5 · selected

In [48]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 0.5

In [49]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.181)


In [50]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 3
n_global = 800000
n_local = 200000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=47,
    local_scale=0.08,
    local_method="truncated",
)

In [51]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [52]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[6.72625e-01 8.22946e-01 5.80000e-04]

Predicted mean:
0.0006069406061667837

Predicted standard deviation:
0.046491564469829424

UCB:
0.023852722841081496


### Option 2: Isotropic GP within a local trust region · alternative

In [53]:
# Week 5 GP model
beta = 0.25

kernel_iso = RBF(
    length_scale=0.1,
    length_scale_bounds=(0.1, 1.0)
)

model_iso = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# Identify current best observation
best_X = X[np.argmax(y)]

# Restrict candidate search to local neighbourhood
local_radius = 0.12

bounds = [
    (max(0, xi - local_radius),
     min(1, xi + local_radius))
    for xi in best_X
]

def negative_local_ucb(x):
    distance = np.linalg.norm(x - best_X)

    # Restrict candidates to a sphere
    if distance > local_radius:
        return 1000 + distance

    mu, std = model_iso.predict(
        np.asarray(x).reshape(1, -1),
        return_std=True
    )

    return -(mu[0] + beta * std[0])

result = differential_evolution(
    negative_local_ucb,
    bounds=bounds,
    seed=42,
    popsize=18,
    maxiter=160,
    tol=1e-9,
    polish=True
)

next_X = np.round(result.x, 6)

mu, std = model_iso.predict(
    next_X.reshape(1, -1),
    return_std=True
)

print("Optimised kernel:", model_iso.kernel_)
print("Week 5 candidate:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])

Optimised kernel: RBF(length_scale=0.181)
Week 5 candidate: [0.435858 0.57608  0.437741]
Predicted mean: 0.001506251283988097
Predicted std: 0.015136524847247692
UCB: 0.0052903824958000206


### Evaluated query

**Input:** `(0.672625, 0.822946, 0.000580)`  
**Output:** `-0.14107932151649005`

In [ ]:
X_week5 = np.array([0.672625, 0.822946, 0.000580])
y_week5 = np.array([-0.14107932151649005])
X_new, y_new = add_evaluation(X, y, X_week5, y_week5)

## Week 6

In [57]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 1 · selected

In [58]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 1

In [59]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.1, 1.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.136)


In [60]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 3
n_global = 800000
n_local = 200000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=48,
    local_scale=0.08,
    local_method="truncated",
)

In [61]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [62]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.389785 0.601897 0.442863]

Predicted mean:
-0.00750360544964096

Predicted standard deviation:
0.046034376012940324

UCB:
0.038530770563299364


### Evaluated query

**Input:** `(0.389785, 0.601897, 0.442863)`  
**Output:** `-0.0092484857317409`

In [ ]:
X_week6 = np.array([0.389785, 0.601897, 0.442863])
y_week6 = np.array([-0.0092484857317409])
X_new, y_new = add_evaluation(X, y, X_week6, y_week6)

## Week 7

In [66]:
# All observations through Week 6
X = X_new
y = y_new

assert X.shape == (21, 3)
assert y.shape == (21,)

# Selected hyperparameters
kernel = Matern(
    length_scale=0.2,
    length_scale_bounds=(0.02, 3.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)
print("Optimised kernel:", model.kernel_)

# GP-UCB
beta = 1.0
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(point):
    point = np.asarray(point)
    distance = tree.query(point)[0]

    # Exclude points too close to observations
    if distance < min_distance:
        return 1000 + 1000 * (min_distance - distance)

    mu, std = model.predict(
        point.reshape(1, -1),
        return_std=True
    )

    return -(mu[0] + beta * std[0])

# Optimise the acquisition function
result = differential_evolution(
    negative_ucb,
    bounds=[(0, 1)] * 3,
    seed=49,
    popsize=15,
    maxiter=120,
    tol=1e-6,
    polish=True
)

next_X = result.x

mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

print("Week 7 recommended input:", np.round(next_X, 6))
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print("Nearest observation distance:", tree.query(next_X)[0])

Optimised kernel: Matern(length_scale=0.209, nu=1.5)
Week 7 recommended input: [0.437511 0.541824 0.517868]
Predicted mean: -0.009641255919747488
Predicted std: 0.03155445298437581
UCB: 0.02191319706462832
Nearest observation distance: 0.08308549076732691


### Evaluated query

**Input:** `(0.437511, 0.541824, 0.517868)`  
**Output:** `-0.013588785694437671`

In [ ]:
X_week7 = np.array([0.437511, 0.541824, 0.517868])
y_week7 = np.array([-0.013588785694437671])
X_new, y_new = add_evaluation(X, y, X_week7, y_week7)

## Week 8

In [70]:
# All 22 observations, including Week 7
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (22, 3)
assert y.shape == (22,)

# Selected GP model
kernel = Matern(
    length_scale=0.2,
    length_scale_bounds=(0.02, 4.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
)
print("Optimised kernel:", model.kernel_)

# GP-UCB acquisition
beta = 1.0
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(point):
    point = np.asarray(point)
    distance = tree.query(point)[0]

    # Exclude candidates near existing observations
    if distance < min_distance:
        return 10 + 100 * (min_distance - distance)

    mu, std = model.predict(
        point.reshape(1, -1),
        return_std=True
    )

    return -(mu[0] + beta * std[0])

# Search acquisition function using multiple seeds
solutions = []

for seed in [42, 48, 49, 55]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 3,
        seed=seed,
        popsize=15,
        maxiter=120,
        tol=1e-7,
        polish=True
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError("No feasible candidate found.")

best_result = min(solutions, key=lambda r: r.fun)
next_X = np.round(best_result.x, 6)

mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

print("Week 8 query:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print("Nearest distance:", tree.query(next_X)[0])

Optimised kernel: Matern(length_scale=0.223, nu=1.5)
Week 8 query: [0.432308 0.529282 0.412299]
Predicted mean: -0.0038113165955261374
Predicted std: 0.02203018771465337
UCB: 0.018218871119127233
Nearest distance: 0.06948001560161017


### Evaluated query

**Input:** `(0.432308, 0.529282, 0.412299)`  
**Output:** `-0.007631641370596846`

In [ ]:
X_week8 = np.array([0.432308, 0.529282, 0.412299])
y_week8 = np.array([-0.007631641370596846])
X_new, y_new = add_evaluation(X, y, X_week8, y_week8)

## Week 9

In [74]:
# All observations through Week 8
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (23, 3)
assert y.shape == (23,)

# Selected global GP
kernel = Matern(
    length_scale=0.2,
    length_scale_bounds=(0.02, 4.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
)

print("Optimised kernel:", model.kernel_)

# Acquisition parameters
beta = 1.0
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(points):
    points = np.asarray(points)

    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = model.predict(
        candidates,
        return_std=True
    )

    distances = tree.query(candidates)[0]

    penalty = 100 * np.maximum(
        0, min_distance - distances
    ) ** 2

    scores = -(ucb_score(mu, std, beta)) + penalty

    return (
        scores if points.ndim == 2
        else scores[0]
    )

# Optimise acquisition using multiple seeds
solutions = []

for seed in [42, 48, 59, 111]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 3,
        seed=seed,
        popsize=15,
        maxiter=160,
        tol=1e-8,
        vectorized=True,
        updating="deferred",
        polish=True
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible acquisition candidate found."
    )

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(best_result.x, 6)

mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

print("Week 9 query:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print(
    "Nearest distance:",
    tree.query(next_X)[0]
)
print("Current best output:", y.max())

Optimised kernel: Matern(length_scale=0.237, nu=1.5)
Week 9 query: [0.360135 0.267653 0.464402]
Predicted mean: -0.03677818761288181
Predicted std: 0.05355537155906265
UCB: 0.016777183946180836
Nearest distance: 0.18707868873564698
Current best output: -0.0001325889446007488


### Evaluated query

**Input:** `(0.360135, 0.267653, 0.464402)`  
**Output:** `-0.03271735233259902`

In [ ]:
X_week9 = np.array([0.360135, 0.267653, 0.464402])
y_week9 = np.array([-0.03271735233259902])
X_new, y_new = add_evaluation(X, y, X_week9, y_week9)

## Week 10

In [78]:
# Observations through Week 9
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (24, 3)
assert y.shape == (24,)
# Fit selected GP
kernel = Matern(
    length_scale=0.2,
    length_scale_bounds=(0.02, 4.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
)

print("Optimised kernel:", model.kernel_)
# Acquisition parameters
beta = 0.75
min_distance = 0.05

tree = cKDTree(X)

def negative_ucb(points):
    points = np.asarray(points)

    candidates = (
        points.T
        if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = model.predict(
        candidates,
        return_std=True
    )

    distances = tree.query(candidates)[0]

    ucb = ucb_score(mu, std, beta)

    penalty = 1000 * np.maximum(
        0,
        min_distance - distances
    )

    scores = -ucb + penalty

    return (
        scores if points.ndim == 2
        else scores[0]
    )
# Optimise GP-UCB
solutions = []

for seed in [42, 59, 111]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 3,
        seed=seed,
        popsize=13,
        maxiter=120,
        tol=1e-7,
        vectorized=True,
        updating="deferred",
        polish=True
    )

    if tree.query(result.x)[0] >= (
        min_distance - 1e-5
    ):
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible acquisition candidate found."
    )

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(
    best_result.x,
    6
)
# Evaluate the selected query
mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

current_best = np.max(y)

prob_improvement = norm.cdf(
    (mu[0] - current_best) / std[0]
)

print("Week 10 query:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("GP-UCB:", mu[0] + beta * std[0])
print(
    "Probability of improvement:",
    prob_improvement
)
print(
    "Nearest-observation distance:",
    tree.query(next_X)[0]
)
print("Current best output:", current_best)

Optimised kernel: Matern(length_scale=0.242, nu=1.5)
Week 10 query: [0.464942 0.606056 0.44849 ]
Predicted mean: -0.0016020715781733114
Predicted std: 0.014882610067274335
GP-UCB: 0.00955988597228244
Probability of improvement: 0.46067305546539306
Nearest-observation distance: 0.05144076178479475
Current best output: -0.0001325889446007488


### Evaluated query

**Input:** `(0.464942, 0.606056, 0.448490)`  
**Output:** `-0.010720393905012877`

In [ ]:
X_week10 = np.array([0.464942, 0.606056, 0.44849])
y_week10 = np.array([-0.010720393905012877])
X_new, y_new = add_evaluation(X, y, X_week10, y_week10)

## Week 11

In [81]:
# Observations through Week 10
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (25, 3)
assert y.shape == (25,)

best_index = np.argmax(y)
best_X = X[best_index]
best_y = y[best_index]

print("Best observed input:", best_X)
print("Best observed output:", best_y)
# Fit Gaussian Process
kernel = Matern(
    length_scale=0.2,
    length_scale_bounds=(0.02, 3.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=5,
    random_state=42
)

print("Optimised kernel:", model.kernel_)
# Local trust region
radius = 0.14
min_distance = 0.02

bounds = [
    (max(0, x - radius), min(1, x + radius))
    for x in best_X
]

tree = cKDTree(X)
# Exploitation-focused acquisition
def negative_mean(point):

    distance = tree.query(point)[0]

    if distance < min_distance:
        return 1 + 5 * (min_distance - distance)

    mu = model.predict(
        np.asarray(point).reshape(1, -1)
    )[0]

    return -mu
# Optimise with multiple seeds
solutions = []

for seed in [42, 59]:

    result = differential_evolution(
        negative_mean,
        bounds=bounds,
        seed=seed,
        popsize=8,
        maxiter=80,
        tol=1e-6,
        polish=True
    )

    solutions.append(result)

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(best_result.x, 6)
# Evaluate selected candidate
mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

beta = 0.5

ucb = mu[0] + beta * std[0]

pi = norm.cdf(
    (mu[0] - best_y) / std[0]
)

print("\nWeek 11 recommended input:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("GP-UCB:", ucb)
print("Probability of improvement:", pi)
print("Nearest distance:", tree.query(next_X)[0])

Best observed input: [0.482994 0.557918 0.450225]
Best observed output: -0.0001325889446007488
Optimised kernel: Matern(length_scale=0.249, nu=1.5)

Week 11 recommended input: [0.47708  0.538842 0.451282]
Predicted mean: 0.0004920148440196032
Predicted std: 0.005934977151596616
GP-UCB: 0.003459503419817911
Probability of improvement: 0.5419077683461448
Nearest distance: 0.019999660522118847


### Evaluated query

**Input:** `(0.477080, 0.538842, 0.451282)`  
**Output:** `-0.007762438965067155`

In [ ]:
X_week11 = np.array([0.47708 , 0.538842, 0.451282])
y_week11 = np.array([-0.007762438965067155])
X_new, y_new = add_evaluation(X, y, X_week11, y_week11)

## Week 12

In [84]:
# Observations through Week 11
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (26, 3)
assert y.shape == (26,)

best_idx = np.argmax(y)
best_X = X[best_idx]
best_y = y[best_idx]

print("Current best input:", best_X)
print("Current best output:", best_y)
# Refit the selected GP
kernel = Matern(
    length_scale=0.2,
    length_scale_bounds=(0.02, 3.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=6,
    random_state=42
)

print("Optimised kernel:", model.kernel_)
# Define local trust region
radius = 0.02
min_distance = 0.008

bounds = [
    (max(0, x - 0.06), min(1, x + 0.06))
    for x in best_X
]

tree = cKDTree(X)
# Optimise predicted mean
def negative_acquisition(point):

    point = np.asarray(point)

    mu = model.predict(
        point.reshape(1, -1)
    )[0]

    nearest_distance = tree.query(point)[0]
    local_distance = np.linalg.norm(point - best_X)

    # Penalise overly close or distant candidates
    penalty = (
        20 * max(0, min_distance - nearest_distance)**2
        + 20 * max(0, local_distance - radius)**2
    )

    return -mu + penalty
# Differential evolution
result = differential_evolution(
    negative_acquisition,
    bounds=bounds,
    seed=42,
    popsize=12,
    maxiter=150,
    tol=1e-8,
    polish=True
)

next_X = np.round(result.x, 6)
# Evaluate candidate
mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

beta = 0.5
ucb = mu[0] + beta * std[0]

print("\nWeek 12 proposed input:")
print(" ".join(f"{v:.6f}" for v in next_X))

print("\nPredicted mean:", mu[0])
print("Predicted std:", std[0])
print("GP-UCB:", ucb)

print(
    "Nearest observation distance:",
    tree.query(next_X)[0]
)

print(
    "Distance from current best:",
    np.linalg.norm(next_X - best_X)
)

print("Current best actual output:", best_y)

Current best input: [0.482994 0.557918 0.450225]
Current best output: -0.0001325889446007488
Optimised kernel: Matern(length_scale=0.238, nu=1.5)

Week 12 proposed input:
0.488961 0.568021 0.447874

Predicted mean: 0.0011821851454066334
Predicted std: 0.0032120264320349705
GP-UCB: 0.0027881983614241185
Nearest observation distance: 0.011966741369311841
Distance from current best: 0.011966741369311841
Current best actual output: -0.0001325889446007488


### Evaluated query

**Input:** `(0.488961, 0.568021, 0.447874)`  
**Output:** `-0.00816456391237672`

In [ ]:
X_week12 = np.array([0.488961, 0.568021, 0.447874])
y_week12 = np.array([-0.00816456391237672])
X_new, y_new = add_evaluation(X, y, X_week12, y_week12)

## Week 13

In [87]:
# Observations through Week 12
X_all = np.asarray(X_new, dtype=float)
y_all = np.asarray(y_new, dtype=float).ravel()

assert X_all.shape == (27, 3)
assert y_all.shape == (27,)

best_idx = np.argmax(y_all)

best_X = X_all[best_idx]
best_y = y_all[best_idx]

print("Current best input:", best_X)
print("Current best output:", best_y)
# Final Week 13 candidate
next_X = np.array([
    0.481800,
    0.556700,
    0.450500
])
# Fit complementary models
def fit_gp(target):
    model = GaussianProcessRegressor(
        kernel=Matern(
            length_scale=0.2,
            length_scale_bounds=(0.01, 3.0),
            nu=1.5
        ),
        alpha=1e-8,
        normalize_y=True,
        n_restarts_optimizer=2,
        random_state=42
    )

    model.fit(X_all, target)
    return model

raw_gp = fit_gp(y_all)

# These transformations require negative outputs
assert np.all(y_all < 0)

sqrt_gp = fit_gp(np.sqrt(-y_all))
log_gp = fit_gp(np.log(-y_all))
# Compare predictions
candidate = next_X.reshape(1, -1)

pred_raw = raw_gp.predict(candidate)[0]

pred_sqrt = (
    -sqrt_gp.predict(candidate)[0]**2
)

pred_log = (
    -np.exp(log_gp.predict(candidate)[0])
)

print("\nModel predictions:")
print(f"Raw GP:         {pred_raw:.9f}")
print(f"Square-root GP: {pred_sqrt:.9f}")
print(f"Log GP:         {pred_log:.9f}")
# Check distance from observations
tree = cKDTree(X_all)

nearest_distance = tree.query(next_X)[0]

distance_from_best = np.linalg.norm(
    next_X - best_X
)

print("\nNearest observation distance:",
      nearest_distance)

print("Distance from current best:",
      distance_from_best)
# Print final coordinates
print("\nFINAL WEEK 13 SUBMISSION:")

print(
    " ".join(
        f"{v:.6f}" for v in next_X
    )
)

Current best input: [0.482994 0.557918 0.450225]
Current best output: -0.0001325889446007488

Model predictions:
Raw GP:         0.000505242
Square-root GP: -0.000052330
Log GP:         -0.000151871

Nearest observation distance: 0.0017276530322955696
Distance from current best: 0.0017276530322955696

FINAL WEEK 13 SUBMISSION:
0.481800 0.556700 0.450500


### Week 13 evaluated result

**Input:** `(0.481800, 0.556700, 0.450500)`  
**Output:** `-0.00012335097135795844`

The query-selection calculations above use results available through Week 12. The full record is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv).